In [0]:
from pyspark.sql.functions import col, trim, lower

# 1. Data read from bronze layer
df_bronze_read = spark.table("workspace.my_database.customers_bronze")

# 2. Column names (change spaces to underscore and make lowercase)
df_silver = df_bronze_read
for col_name in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(col_name, col_name.replace(" ", "_").lower())

# 3. Data cleaning steps: Drop duplicates AND drop rows with nulls in specific columns
# Yahan humne "email" aur "phone" ko subset mein add kar diya hai
df_silver_cleaned = df_silver \
    .dropDuplicates() \
    .na.drop(subset=["customer_id", "email", "phone"])

# 4. Cast 'phone' to String (Taki aage join ya formatting mein asani ho)
df_silver_cleaned = df_silver_cleaned.withColumn("phone", col("phone").cast("string"))

# 5. Save the cleaned data to the Silver table
df_silver_cleaned.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.my_database.customers_silver")

print("Silver Layer loaded successfully! (Rows with null emails/phones have been dropped)")
display(df_silver_cleaned)